# Market Signals and Market Activity Generator

This notebook generates two synthetic enrichment tables:

1. **`factMarketSignals.csv`** at `year_month × product_group × region_id` grain.
2. **`factMarketActivities.csv`** at `year_month × product_id × region_id` grain.

The generated data uses the project's real calendar, products, product hierarchy, regions, lifecycle stages, prices, margins, inventory conditions, CRM intensity and pipeline interest. All keys and data types are compatible with the existing star-schema tables.

## Imports and reproducible configuration

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

RANDOM_SEED = 42
MARKET_SEED = RANDOM_SEED + 20000
N_SALES_ROWS_IF_REBUILT = 120000

BASE_DATA_DIR = Path("../data/sql_input")
INPUT_DIR_PREPROCESSED = Path("../data/preprocessed")
OUTPUT_DIR = Path("../data/new_generated_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_BASE_FILES = {
    "dimDate": "dimDate.csv",
    "dimRegion": "dimRegion.csv",
    "dimProduct": "dimProduct.csv",
    "dimCustomer": "dimCustomer.csv",
    "factSales": "factSales.csv",
    "factInventory": "factInventory.csv",
    "factCRMActivities": "factCRMActivities.csv",
    "factPipeline": "factPipeline.csv"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input sql data directory: {BASE_DATA_DIR.resolve()}")
print(f"Input processed data directory: {INPUT_DIR_PREPROCESSED.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Input sql data directory: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\early-warning\data\sql_input
Input processed data directory: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\early-warning\data\preprocessed
Output directory: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\early-warning\data\new_generated_data


## Load the existing project data

In [ ]:
# Load the required source tables when all expected CSVs are present
def load_existing_base_data(data_dir: Path) -> dict[str, pd.DataFrame] | None:
    paths = {name: data_dir / filename for name, filename in REQUIRED_BASE_FILES.items()}
    if not all(path.exists() for path in paths.values()):
        return None

    tables = {
        name: pd.read_csv(path, sep=",", decimal=",", low_memory=False)
        for name, path in paths.items()}
    print("Loaded existing revenue-intelligence CSV files.")
    return tables

# Execute function-definition cells from the source notebook, excluding its main run cell
def load_generator_functions(notebook_path: Path) -> dict:
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    namespace = {"__name__": "revenue_intelligence_base_generator"}

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        if not source.strip() or "if __name__ == \"__main__\"" in source:
            continue
        exec(compile(source, str(notebook_path), "exec"), namespace)

    if "generate_dataset" not in namespace:
        raise ValueError("The attached notebook does not define generate_dataset().")
    return namespace


# Run the attached generator without exporting any of its base tables
def build_base_data_in_memory(notebook_path: Path) -> dict[str, pd.DataFrame]:
    namespace = load_generator_functions(notebook_path)
    original_to_csv = pd.DataFrame.to_csv

    try:
        pd.DataFrame.to_csv = lambda self, *args, **kwargs: None
        generated_tables = namespace["generate_dataset"](
            output_dir=BASE_DATA_DIR,
            n_sales_rows=N_SALES_ROWS_IF_REBUILT,
            seed=RANDOM_SEED)
    finally:
        pd.DataFrame.to_csv = original_to_csv

    missing_tables = set(REQUIRED_BASE_FILES) - set(generated_tables)
    if missing_tables:
        raise ValueError(f"Generated base data is missing tables: {sorted(missing_tables)}")

    print(f"Recreated project tables in memory using: {notebook_path.name}")
    return {name: generated_tables[name].copy() for name in REQUIRED_BASE_FILES}

base_tables = load_existing_base_data(BASE_DATA_DIR)
if base_tables is None:
    base_tables = build_base_data_in_memory(resolve_generator_path())

dim_date = base_tables["dimDate"].copy()
dim_region = base_tables["dimRegion"].copy()
dim_product = base_tables["dimProduct"].copy()
dim_customer = base_tables["dimCustomer"].copy()
fact_sales = base_tables["factSales"].copy()
fact_inventory = base_tables["factInventory"].copy()
fact_crm = base_tables["factCRMActivities"].copy()
fact_pipeline = base_tables["factPipeline"].copy()

base_summary = pd.DataFrame({
    "table": list(REQUIRED_BASE_FILES),
    "rows": [len(base_tables[name]) for name in REQUIRED_BASE_FILES],
    "columns": [base_tables[name].shape[1] for name in REQUIRED_BASE_FILES]})
base_summary

Loaded existing revenue-intelligence CSV files.


,table,rows,columns
0,dimDate,4018,19
1,dimRegion,10,8
2,dimProduct,200,17
3,dimCustomer,1200,10
4,factSales,120000,51
5,factInventory,10,11
6,factCRMActivities,35000,9
7,factPipeline,15000,19


## Standardize keys and build internal calibration drivers

The two outputs use the same key conventions as the existing facts and dimensions:

- `year_month`: `YYYY-MM` string matching `factSales`, `factInventory`, `factForecast`, and `factCosts`;
- `product_id`: integer matching `dimProduct.product_id`;
- `product_group`: matching `dimProduct.product_group` and `factPipeline.product_group`;
- `region_id`: integer matching `dimRegion.region_id`.

Inventory, CRM, and pipeline aggregations below are internal drivers only and are not exported.

In [ ]:
required_columns = {
    "dimDate": {"Date", "YearMonth"},
    "dimRegion": {"region_id", "market_growth_factor", "margin_factor", "fx_to_eur"},
    "dimProduct": {
        "product_id", "product_family", "product_group", "lifecycle_stage",
        "base_list_price_eur", "target_margin_pct", "product_growth_factor"},
    "dimCustomer": {"customer_id", "region_id"},
    "factSales": {"year_month", "product_id", "region_id"},
    "factInventory": {
        "year_month", "product_id", "region_id", "opening_stock_units",
        "production_units", "ending_stock_units", "stockout_flag"},
    "factCRMActivities": {
        "date", "customer_id", "activity_id", "activity_minutes", "customer_health_score"},
    "factPipeline": {
        "created_date", "customer_id", "product_group", "opportunity_id",
        "weighted_pipeline_eur", "win_probability"}}

for table_name, expected_columns in required_columns.items():
    missing = expected_columns - set(base_tables[table_name].columns)
    assert not missing, f"{table_name} is missing columns: {sorted(missing)}"

assert dim_region["region_id"].is_unique
assert dim_product["product_id"].is_unique
assert dim_customer["customer_id"].is_unique
assert set(fact_sales["product_id"]).issubset(set(dim_product["product_id"]))
assert set(fact_sales["region_id"]).issubset(set(dim_region["region_id"]))

dim_date["YearMonth"] = dim_date["YearMonth"].astype(str)
dim_date["YearMonth"] = normalize_year_month(
    dim_date["YearMonth"],
    "dimDate.YearMonth")

fact_sales["year_month"] = normalize_year_month(
    fact_sales["year_month"],
    "factSales.year_month")

fact_inventory["year_month"] = normalize_year_month(
    fact_inventory["year_month"],
    "factInventory.year_month")

fact_crm["year_month"] = normalize_year_month(
    fact_crm["date"],
    "factCRMActivities.date")

fact_pipeline["year_month"] = normalize_year_month(
    fact_pipeline["created_date"],
    "factPipeline.created_date")

months = sorted(
    set(dim_date["YearMonth"].dropna())
    | set(fact_sales["year_month"].dropna())
    | set(fact_inventory["year_month"].dropna()))
region_ids = sorted(dim_region["region_id"].astype(int).unique())
product_ids = sorted(dim_product["product_id"].astype(int).unique())

# ------------------------------------------------------------------
# Normalize data types before aggregations
# ------------------------------------------------------------------

def convert_to_numeric(series, percent=False):
    """
    Converts numeric strings, currency-formatted values and percentages
    into numeric values.

    Examples:
        "1,250.50" -> 1250.50
        "1.250,50" -> 1250.50
        "35%"      -> 0.35 when percent=True
    """
    values = series.astype("string").str.strip()
    percent_mask = values.str.contains("%", na=False)

    # Remove currency symbols and spaces
    values = values.str.replace(r"[€$£\s]", "", regex=True)

    comma_position = values.str.rfind(",")
    dot_position = values.str.rfind(".")

    has_comma = values.str.contains(",", regex=False, na=False)
    has_dot = values.str.contains(".", regex=False, na=False)
    has_both = has_comma & has_dot

    # European format: 1.250,50
    european_format = has_both & (comma_position > dot_position)
    values.loc[european_format] = (
        values.loc[european_format]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False))

    # International format: 1,250.50
    international_format = has_both & ~european_format
    values.loc[international_format] = (
        values.loc[international_format]
        .str.replace(",", "", regex=False))

    # Decimal comma without a dot: 250,50
    comma_only = has_comma & ~has_dot
    values.loc[comma_only] = (
        values.loc[comma_only]
        .str.replace(",", ".", regex=False))

    numeric_values = pd.to_numeric(values, errors="coerce")

    if percent:
        numeric_values.loc[percent_mask] = (
            numeric_values.loc[percent_mask] / 100)

    return numeric_values

def normalize_year_month(series, column_name):
    """Convert dates and month representations to YYYY-MM."""
    raw = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True))

    parsed = pd.Series(
        pd.NaT,
        index=series.index,
        dtype="datetime64[ns]")

    # Format such as 202501
    yyyymm_mask = raw.str.fullmatch(r"\d{6}", na=False)

    parsed.loc[yyyymm_mask] = pd.to_datetime(
        raw.loc[yyyymm_mask],
        format="%Y%m",
        errors="coerce")

    # Formats such as 2025-01, 2025-01-01 or full dates
    parsed.loc[~yyyymm_mask] = pd.to_datetime(
        raw.loc[~yyyymm_mask],
        errors="coerce")

    if parsed.isna().any():
        invalid_examples = (
            raw.loc[parsed.isna()]
            .drop_duplicates()
            .head(10)
            .tolist())

        raise ValueError(
            f"{column_name} contains invalid month values: "
            f"{invalid_examples}")

    return parsed.dt.to_period("M").astype(str) 

def normalize_product_group(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.casefold())

for dataframe in [
    dim_region,
    fact_sales,
    fact_inventory,
    fact_crm,
    fact_pipeline,
]:
    if "region_id" in dataframe.columns:
        dataframe["region_id"] = pd.to_numeric(
            dataframe["region_id"],
            errors="raise",
        ).astype(int)

for dataframe in [
    dim_product,
    fact_sales,
    fact_inventory,
]:
    if "product_id" in dataframe.columns:
        dataframe["product_id"] = pd.to_numeric(
            dataframe["product_id"],
            errors="raise",
        ).astype(int)

dim_product["product_group"] = normalize_product_group(
    dim_product["product_group"]
)

fact_pipeline["product_group"] = normalize_product_group(
    fact_pipeline["product_group"]
)

# Product columns used in aggregations
dim_product["base_list_price_eur"] = convert_to_numeric(
    dim_product["base_list_price_eur"]
)
dim_product["target_margin_pct"] = convert_to_numeric(
    dim_product["target_margin_pct"],
    percent=True,
)
dim_product["product_growth_factor"] = convert_to_numeric(
    dim_product["product_growth_factor"]
)

# CRM columns
fact_crm["activity_minutes"] = convert_to_numeric(
    fact_crm["activity_minutes"]
)
fact_crm["customer_health_score"] = convert_to_numeric(
    fact_crm["customer_health_score"]
)

# Pipeline columns
fact_pipeline["weighted_pipeline_eur"] = convert_to_numeric(
    fact_pipeline["weighted_pipeline_eur"]
)
fact_pipeline["win_probability"] = convert_to_numeric(
    fact_pipeline["win_probability"],
    percent=True,
)

# Inventory columns
for column in [
    "opening_stock_units",
    "production_units",
    "ending_stock_units",
]:
    fact_inventory[column] = convert_to_numeric(
        fact_inventory[column]
    )

# Check whether conversion produced invalid values
numeric_validation = {
    "dimProduct.base_list_price_eur": dim_product["base_list_price_eur"],
    "dimProduct.target_margin_pct": dim_product["target_margin_pct"],
    "dimProduct.product_growth_factor": dim_product["product_growth_factor"],
    "factCRMActivities.activity_minutes": fact_crm["activity_minutes"],
    "factCRMActivities.customer_health_score": fact_crm["customer_health_score"],
    "factPipeline.weighted_pipeline_eur": fact_pipeline["weighted_pipeline_eur"],
    "factPipeline.win_probability": fact_pipeline["win_probability"],
}

for column_name, values in numeric_validation.items():
    missing_count = values.isna().sum()

    if missing_count:
        print(
            f"Warning: {column_name} contains "
            f"{missing_count:,} missing or invalid numeric values."
        )

product_info = dim_product[[
    "product_id",
    "product_family",
    "product_group",
    "lifecycle_stage",
    "base_list_price_eur",
    "target_margin_pct",
    "product_growth_factor",
]].copy()

product_info["product_id"] = pd.to_numeric(
    product_info["product_id"],
    errors="raise",
).astype(int)

group_info = (
    product_info
    .groupby(
        ["product_family", "product_group"],
        as_index=False,
        dropna=False,
    )
    .agg(
        product_count=("product_id", "nunique"),
        median_list_price_eur=(
            "base_list_price_eur",
            "median",
        ),
        average_target_margin_pct=(
            "target_margin_pct",
            "mean",
        ),
        average_product_growth_factor=(
            "product_growth_factor",
            "mean",
        ),
        decline_product_share=(
            "lifecycle_stage",
            lambda values: (
                values.astype("string")
                .str.strip()
                .str.casefold()
                .eq("decline")
                .mean()
            ),
        ),
    )
)

assert group_info["product_group"].is_unique, (
    "product_group is not unique across product families. "
    "Use both product_family and product_group as the grouping key."
)

customer_region = dim_customer[["customer_id", "region_id"]].copy()
customer_region["region_id"] = customer_region["region_id"].astype(int)

# Regional CRM activity driver.
crm_base = fact_crm.drop(
    columns=["region_id"],
    errors="ignore",
).copy()

crm_region_month = (
    crm_base
    .merge(
        customer_region,
        on="customer_id",
        how="left",
        validate="many_to_one",
    )
    .groupby(
        ["year_month", "region_id"],
        as_index=False,
    )
    .agg(
        crm_activity_count=("activity_id", "nunique"),
        crm_activity_minutes=("activity_minutes", "sum"),
        average_customer_health=("customer_health_score", "mean"),
    )
)

region_month_panel = pd.MultiIndex.from_product(
    [months, region_ids], names=["year_month", "region_id"]
).to_frame(index=False)
crm_region_month = region_month_panel.merge(
    crm_region_month,
    on=["year_month", "region_id"],
    how="left",
).fillna({"crm_activity_count": 0, "crm_activity_minutes": 0, "average_customer_health": 65})
monthly_crm_average = crm_region_month.groupby("year_month")["crm_activity_count"].transform("mean")
crm_region_month["regional_crm_activity_index"] = (
    100 * crm_region_month["crm_activity_count"] / monthly_crm_average.replace(0, np.nan)
).fillna(100).clip(20, 250).round(2)

# Product-group pipeline driver.
# Region assignment comes from dimCustomer.

pipeline_base = fact_pipeline.drop(
    columns=["region_id"],
    errors="ignore",
).copy()

pipeline_enriched = pipeline_base.merge(
    customer_region,
    on="customer_id",
    how="left",
    validate="many_to_one",
)

# Validate customer-to-region mapping
missing_region_count = pipeline_enriched["region_id"].isna().sum()

if missing_region_count:
    missing_customers = (
        pipeline_enriched.loc[
            pipeline_enriched["region_id"].isna(),
            "customer_id",
        ]
        .drop_duplicates()
        .head(10)
        .tolist()
    )

    raise ValueError(
        f"{missing_region_count:,} pipeline rows could not be assigned "
        f"to a region. Example customer IDs: {missing_customers}"
    )

pipeline_enriched["region_id"] = (
    pd.to_numeric(
        pipeline_enriched["region_id"],
        errors="raise",
    )
    .astype(int)
)

pipeline_group_month = (
    pipeline_enriched
    .groupby(
        ["year_month", "product_group", "region_id"],
        as_index=False,
        dropna=False,
    )
    .agg(
        opportunity_count=(
            "opportunity_id",
            "nunique",
        ),
        weighted_pipeline_eur=(
            "weighted_pipeline_eur",
            "sum",
        ),
        average_win_probability=(
            "win_probability",
            "mean",
        ),
    )
)

signal_panel = pd.MultiIndex.from_product(
    [
        months,
        sorted(group_info["product_group"].dropna().unique()),
        region_ids,
    ],
    names=[
        "year_month",
        "product_group",
        "region_id",
    ],
).to_frame(index=False)

pipeline_group_month = signal_panel.merge(
    pipeline_group_month,
    on=[
        "year_month",
        "product_group",
        "region_id",
    ],
    how="left",
    validate="one_to_one",
)

pipeline_group_month = pipeline_group_month.fillna({
    "opportunity_count": 0,
    "weighted_pipeline_eur": 0,
    "average_win_probability": 0,
})

monthly_pipeline_average = (
    pipeline_group_month
    .groupby("year_month")["opportunity_count"]
    .transform("mean")
)

pipeline_group_month["pipeline_interest_index"] = (
    100
    * pipeline_group_month["opportunity_count"]
    / monthly_pipeline_average.replace(0, np.nan)
).fillna(0).clip(0, 300).round(2)

signal_panel = pd.MultiIndex.from_product(
    [months, sorted(group_info["product_group"]), region_ids],
    names=["year_month", "product_group", "region_id"],
).to_frame(index=False)
pipeline_group_month = signal_panel.merge(
    pipeline_group_month,
    on=["year_month", "product_group", "region_id"],
    how="left",
).fillna({"opportunity_count": 0, "weighted_pipeline_eur": 0, "average_win_probability": 0})
monthly_pipeline_average = pipeline_group_month.groupby("year_month")[
    "opportunity_count"
].transform("mean")
pipeline_group_month["pipeline_interest_index"] = (
    100
    * pipeline_group_month["opportunity_count"]
    / monthly_pipeline_average.replace(0, np.nan)
).fillna(0).clip(0, 300).round(2)

# Product-group supply driver from the existing inventory fact.
inventory_enriched = fact_inventory.merge(
    product_info[["product_id", "product_group"]],
    on="product_id",
    how="left",
    validate="many_to_one",
)
inventory_enriched["stockout_flag_numeric"] = (
    inventory_enriched["stockout_flag"].astype(str).str.lower().isin(["true", "1"])
).astype(int)
inventory_enriched["available_stock_ratio"] = (
    inventory_enriched["ending_stock_units"]
    / (
        inventory_enriched["opening_stock_units"]
        + inventory_enriched["production_units"]
    ).replace(0, np.nan)
).fillna(0).clip(0, 1)

inventory_group_month = (
    inventory_enriched.groupby(["year_month", "product_group", "region_id"], as_index=False)
    .agg(
        stockout_rate=("stockout_flag_numeric", "mean"),
        average_available_stock_ratio=("available_stock_ratio", "mean"),
    )
)
inventory_group_month = signal_panel.merge(
    inventory_group_month,
    on=["year_month", "product_group", "region_id"],
    how="left",
).fillna({"stockout_rate": 0, "average_available_stock_ratio": 0.50})
inventory_group_month["supply_pressure_index"] = (
    25
    + 65 * inventory_group_month["stockout_rate"]
    + 35 * (1 - inventory_group_month["average_available_stock_ratio"])
).clip(5, 100).round(2)

print(f"Months: {len(months)} ({months[0]} to {months[-1]})")
print(f"Products: {len(product_ids)}")
print(f"Product groups: {group_info['product_group'].nunique()}")
print(f"Regions: {len(region_ids)}")

Months: 132 (2020-01 to 2030-12)
Products: 200
Product groups: 10
Regions: 10


## Generate enriched `market_signals`

The demand logic follows the source generator's seasonality—Q4 strength and July/August slowdown—and uses the existing regional and product growth factors. It adds persistent competitor pressure, macro movement, supply pressure, pipeline interest, demand shocks, market growth, and a composite opportunity score.

Same-month sales and revenue are not used to construct `market_demand_index`, preventing direct leakage from the target fact.

In [ ]:
def source_seasonality_index(year_month: str) -> float:
    """Return the same seasonal pattern used by the attached sales generator."""
    month_number = int(year_month[-2:])
    if month_number in [10, 11, 12]:
        return 130.0
    if month_number in [7, 8]:
        return 82.0
    return 100.0


def create_market_signals(
    groups: pd.DataFrame,
    regions: pd.DataFrame,
    months: list[str],
    pipeline_driver: pd.DataFrame,
    supply_driver: pd.DataFrame,
    rng: np.random.Generator,
) -> pd.DataFrame:
    """Generate external market signals aligned to product-group and region keys."""
    pipeline_lookup = pipeline_driver.set_index(
        ["year_month", "product_group", "region_id"]
    )["pipeline_interest_index"]
    supply_lookup = supply_driver.set_index(
        ["year_month", "product_group", "region_id"]
    )["supply_pressure_index"]
    region_lookup = regions.set_index("region_id").to_dict("index")

    macro_by_month = {}
    for month_index, year_month in enumerate(months):
        macro_by_month[year_month] = float(np.clip(
            100
            + 0.14 * month_index
            + 3.5 * np.sin(month_index / 6.5)
            + rng.normal(0, 1.1),
            88,
            120,
        ))

    rows = []
    for _, group in groups.sort_values("product_group").iterrows():
        for region_id in sorted(region_lookup):
            region = region_lookup[region_id]
            competitor_state = rng.uniform(35, 74)

            for month_index, year_month in enumerate(months):
                seasonality_index = source_seasonality_index(year_month)
                macro_index = macro_by_month[year_month]
                years_elapsed = month_index / 12

                competitor_state = np.clip(
                    0.84 * competitor_state + 0.16 * 55 + rng.normal(0, 3.8),
                    10,
                    95,
                )
                pipeline_index = float(
                    pipeline_lookup.loc[(year_month, group["product_group"], region_id)]
                )
                supply_pressure = float(
                    supply_lookup.loc[(year_month, group["product_group"], region_id)]
                )

                shock_flag = int(rng.random() < 0.032)
                if shock_flag:
                    shock_multiplier = rng.choice(
                        [rng.uniform(0.76, 0.90), rng.uniform(1.10, 1.24)],
                        p=[0.68, 0.32],
                    )
                else:
                    shock_multiplier = 1.0

                region_growth = float(region["market_growth_factor"]) ** years_elapsed
                product_growth = float(group["average_product_growth_factor"]) ** years_elapsed
                lifecycle_factor = max(
                    0.72,
                    1 - 0.18 * float(group["decline_product_share"]) * years_elapsed)

                demand_index = (
                    100
                    * (seasonality_index / 100)
                    * (macro_index / 100)
                    * region_growth
                    * product_growth
                    * lifecycle_factor
                    * shock_multiplier
                    * rng.lognormal(0, 0.035))
                demand_index = float(np.clip(demand_index, 42, 180))

                opportunity_score = float(np.clip(
                    0.40 * demand_index
                    + 0.22 * (100 - competitor_state)
                    + 0.15 * (100 - supply_pressure)
                    + 0.23 * min(pipeline_index, 100),
                    0,
                    100))

                rows.append({
                    "year_month": year_month,
                    "product_family": group["product_family"],
                    "product_group": group["product_group"],
                    "region_id": int(region_id),
                    "market_demand_index": round(demand_index, 2),
                    "competitor_pressure_index": round(float(competitor_state), 2),
                    "seasonality_index": round(seasonality_index, 2),
                    "macro_business_index": round(macro_index, 2),
                    "supply_pressure_index": round(supply_pressure, 2),
                    "pipeline_interest_index": round(pipeline_index, 2),
                    "demand_shock_flag": shock_flag,
                    "market_opportunity_score": round(opportunity_score, 2),
                    "regional_market_growth_factor": round(
                        float(region["market_growth_factor"]), 3)})

    output = pd.DataFrame(rows).sort_values(
        ["product_group", "region_id", "year_month"]
    ).reset_index(drop=True)
    output["market_growth_pct"] = (
        output.groupby(["product_group", "region_id"])["market_demand_index"]
        .pct_change(fill_method=None)
        .mul(100)
        .fillna(0)
        .clip(-50, 50)
        .round(2))

    final_columns = [
        "year_month", "product_family", "product_group", "region_id",
        "market_demand_index", "market_growth_pct",
        "competitor_pressure_index", "seasonality_index",
        "macro_business_index", "supply_pressure_index",
        "pipeline_interest_index", "demand_shock_flag",
        "market_opportunity_score", "regional_market_growth_factor"]
    return output[final_columns]


signal_rng = np.random.default_rng(MARKET_SEED)
market_signals = create_market_signals(
    groups=group_info,
    regions=dim_region,
    months=months,
    pipeline_driver=pipeline_group_month,
    supply_driver=inventory_group_month,
    rng=signal_rng)

print(f"market_signals: {len(market_signals):,} rows × {market_signals.shape[1]} columns")
market_signals.head()

market_signals: 13,200 rows × 14 columns


,year_month,product_family,product_group,region_id,market_demand_index,market_growth_pct,competitor_pressure_index,seasonality_index,macro_business_index,supply_pressure_index,pipeline_interest_index,demand_shock_flag,market_opportunity_score,regional_market_growth_factor
0,2020-01,Accessories,accessory kit,1,97.04,0.00,53.02,100.0,100.26,42.5,0.0,0,57.78,1.06
1,2020-02,Accessories,accessory kit,1,102.56,5.69,52.16,100.0,103.46,42.5,0.0,0,60.17,1.06
2,2020-03,Accessories,accessory kit,1,101.57,-0.97,51.04,100.0,101.86,42.5,0.0,0,60.02,1.06
3,2020-04,Accessories,accessory kit,1,102.22,0.64,49.51,100.0,102.81,42.5,0.0,0,60.62,1.06
4,2020-05,Accessories,accessory kit,1,103.83,1.58,46.25,100.0,102.07,42.5,0.0,0,61.98,1.06


## Generate enriched `market_activity`

The product-level activity table contains campaign and digital engagement data and is calibrated with:

- product price, target margin, family, lifecycle, and growth factor;
- market opportunity and pipeline interest from `market_signals`;
- regional CRM intensity derived from `factCRMActivities`.

It retains the original core measures—campaign flag/spend, website visits, page views, and demo requests—and adds channel, impressions, clicks, CTR, qualified leads, and cost per lead.

In [50]:
# Generate product-region-month campaign and engagement data
def create_market_activity(
    products: pd.DataFrame,
    signals: pd.DataFrame,
    crm_driver: pd.DataFrame,
    months: list[str],
    region_ids: list[int],
    rng: np.random.Generator,
) -> pd.DataFrame:
    signal_lookup = signals.set_index(["year_month", "product_group", "region_id"])
    crm_lookup = crm_driver.set_index(["year_month", "region_id"])[
        "regional_crm_activity_index"]

    price_log = np.log1p(products["base_list_price_eur"].astype(float))
    price_score = (
        (price_log - price_log.min())
        / max(price_log.max() - price_log.min(), 1e-9))
    margin_score = products["target_margin_pct"].astype(float).clip(0, 1)
    products = products.assign(
        _price_score=price_score.to_numpy(),
        _margin_score=margin_score.to_numpy())

    channels = ["Paid Search", "Email", "Partner", "Trade Fair", "Webinar", "Content"]
    channel_probabilities = [0.23, 0.23, 0.18, 0.10, 0.13, 0.13]
    high_consideration_families = {"Medical Devices", "Industrial Tech", "Services"}

    rows = []
    for _, product in products.sort_values("product_id").iterrows():
        high_consideration = product["product_family"] in high_consideration_families
        launch_or_growth = product["lifecycle_stage"] in ["New", "Growth"]

        for region_id in region_ids:
            for year_month in months:
                signal = signal_lookup.loc[(year_month, product["product_group"], region_id)]
                crm_index = float(crm_lookup.loc[(year_month, region_id)])
                pipeline_index = float(signal["pipeline_interest_index"])

                campaign_probability = float(np.clip(
                    0.045
                    + 0.070 * product["_margin_score"]
                    + 0.060 * high_consideration
                    + 0.055 * launch_or_growth
                    + 0.045 * (signal["market_opportunity_score"] >= 58)
                    + 0.035 * (crm_index >= 105)
                    + 0.030 * (pipeline_index >= 110)
                    + 0.020 * signal["demand_shock_flag"],
                    0.04,
                    0.36))
                campaign_flag = int(rng.random() < campaign_probability)

                if campaign_flag:
                    campaign_channel = str(rng.choice(channels, p=channel_probabilities))
                    spend = (
                        rng.gamma(3.0, 820.0)
                        * (0.80 + 1.15 * product["_price_score"])
                        * (0.75 + crm_index / 280))
                    impressions = int(max(
                        100,
                        rng.normal(spend * rng.uniform(13, 29), max(spend * 2.4, 1))))
                    expected_ctr = float(np.clip(
                        0.017
                        + 0.011 * (signal["market_opportunity_score"] / 100)
                        - 0.004 * (signal["competitor_pressure_index"] / 100)
                        + rng.normal(0, 0.002),
                        0.006,
                        0.055))
                    clicks = int(rng.binomial(impressions, expected_ctr))
                else:
                    campaign_channel = "No Active Campaign"
                    spend = 0.0
                    impressions = 0
                    clicks = 0

                organic_visits = (
                    28
                    + 120 * product["_price_score"]
                    + 1.10 * signal["market_demand_index"]
                    + 0.18 * min(pipeline_index, 200)
                    + rng.normal(0, 25))
                website_visits = int(max(
                    0,
                    organic_visits + clicks * rng.uniform(0.80, 1.20)))
                product_page_views = int(max(
                    0,
                    website_visits * rng.uniform(1.25, 2.80)))

                demo_rate = float(np.clip(
                    0.003
                    + 0.007 * high_consideration
                    + 0.003 * launch_or_growth
                    + 0.002 * campaign_flag,
                    0.002,
                    0.020))
                demo_requests = int(rng.binomial(product_page_views, demo_rate))
                click_leads = int(rng.binomial(clicks, 0.055)) if clicks > 0 else 0
                organic_leads = int(rng.binomial(max(website_visits, 0), 0.002))
                marketing_qualified_leads = demo_requests + click_leads + organic_leads

                campaign_ctr_pct = 100 * clicks / impressions if impressions > 0 else 0.0
                cost_per_lead = (
                    spend / marketing_qualified_leads
                    if spend > 0 and marketing_qualified_leads > 0
                    else 0.0)

                rows.append({
                    "year_month": year_month,
                    "product_id": int(product["product_id"]),
                    "region_id": int(region_id),
                    "campaign_flag": campaign_flag,
                    "campaign_channel": campaign_channel,
                    "campaign_spend_eur": round(float(spend), 2),
                    "campaign_impressions": impressions,
                    "campaign_clicks": clicks,
                    "campaign_ctr_pct": round(float(campaign_ctr_pct), 3),
                    "website_visits": website_visits,
                    "product_page_views": product_page_views,
                    "demo_requests": demo_requests,
                    "marketing_qualified_leads": marketing_qualified_leads,
                    "cost_per_lead_eur": round(float(cost_per_lead), 2),
                    "regional_crm_activity_index": round(crm_index, 2),
                    "pipeline_interest_index": round(pipeline_index, 2)})

    return pd.DataFrame(rows).sort_values(
        ["product_id", "region_id", "year_month"]
    ).reset_index(drop=True)

activity_rng = np.random.default_rng(MARKET_SEED + 1)
market_activity = create_market_activity(
    products=product_info,
    signals=market_signals,
    crm_driver=crm_region_month,
    months=months,
    region_ids=region_ids,
    rng=activity_rng)

print(f"market_activity: {len(market_activity):,} rows × {market_activity.shape[1]} columns")
market_activity.head()

market_activity: 264,000 rows × 16 columns


,year_month,product_id,region_id,campaign_flag,campaign_channel,campaign_spend_eur,campaign_impressions,campaign_clicks,campaign_ctr_pct,website_visits,product_page_views,demo_requests,marketing_qualified_leads,cost_per_lead_eur,regional_crm_activity_index,pipeline_interest_index
0,2020-01,1000,1,0,No Active Campaign,0.00,0,0,0.000,252,480,1,1,0.00,100.0,0.0
1,2020-02,1000,1,0,No Active Campaign,0.00,0,0,0.000,239,320,2,2,0.00,100.0,0.0
2,2020-03,1000,1,1,Trade Fair,4557.48,88934,2173,2.443,2242,6216,25,157,29.03,100.0,0.0
3,2020-04,1000,1,0,No Active Campaign,0.00,0,0,0.000,213,380,4,4,0.00,100.0,0.0
4,2020-05,1000,1,1,Content,2492.41,51298,1134,2.211,1186,3038,15,73,34.14,100.0,0.0


## Data quality and join-coverage validation

The assertions confirm complete dimensional coverage, unique composite keys, correct business rules, compatible data types, and 100% enrichment coverage for every sales row.

In [51]:
expected_signal_rows = len(months) * group_info["product_group"].nunique() * len(region_ids)
expected_activity_rows = len(months) * len(product_ids) * len(region_ids)

assert len(market_signals) == expected_signal_rows
assert len(market_activity) == expected_activity_rows
assert not market_signals.duplicated(["year_month", "product_group", "region_id"]).any()
assert not market_activity.duplicated(["year_month", "product_id", "region_id"]).any()
assert market_signals.isna().sum().sum() == 0
assert market_activity.isna().sum().sum() == 0

assert set(market_signals["year_month"]) == set(months)
assert set(market_activity["year_month"]) == set(months)
assert set(market_signals["product_group"]) == set(group_info["product_group"])
assert set(market_activity["product_id"]) == set(product_ids)
assert set(market_signals["region_id"]) == set(region_ids)
assert set(market_activity["region_id"]) == set(region_ids)

assert pd.api.types.is_integer_dtype(market_signals["region_id"])
assert pd.api.types.is_integer_dtype(market_activity["region_id"])
assert pd.api.types.is_integer_dtype(market_activity["product_id"])
assert market_signals["demand_shock_flag"].isin([0, 1]).all()
assert market_activity["campaign_flag"].isin([0, 1]).all()
assert (
    market_activity.loc[market_activity["campaign_flag"] == 0, "campaign_spend_eur"] == 0
).all()
assert (
    market_activity.loc[market_activity["campaign_flag"] == 0, "campaign_impressions"] == 0
).all()
assert (market_activity.select_dtypes(include=np.number) >= 0).all().all()

# Confirm that every existing sales row can receive both enrichments.
sales_keys = fact_sales[["year_month", "product_id", "region_id"]].merge(
    product_info[["product_id", "product_group"]],
    on="product_id",
    how="left",
    validate="many_to_one")

signal_coverage = sales_keys.merge(
    market_signals[[
        "year_month", "product_group", "region_id", "market_demand_index"
    ]],
    on=["year_month", "product_group", "region_id"],
    how="left"
)["market_demand_index"].notna().mean()

activity_coverage = sales_keys.merge(
    market_activity[["year_month", "product_id", "region_id", "website_visits"]],
    on=["year_month", "product_id", "region_id"],
    how="left"
)["website_visits"].notna().mean()

assert signal_coverage == 1.0
assert activity_coverage == 1.0

quality_summary = pd.DataFrame({
    "dataset": ["market_signals", "market_activity"],
    "rows": [len(market_signals), len(market_activity)],
    "columns": [market_signals.shape[1], market_activity.shape[1]],
    "duplicate_keys": [
        int(market_signals.duplicated(["year_month", "product_group", "region_id"]).sum()),
        int(market_activity.duplicated(["year_month", "product_id", "region_id"]).sum())],
    "missing_values": [
        int(market_signals.isna().sum().sum()),
        int(market_activity.isna().sum().sum())],
    "sales_join_coverage_pct": [signal_coverage * 100, activity_coverage * 100]})

print("All project-alignment and data-quality checks passed.")
quality_summary

All project-alignment and data-quality checks passed.


,dataset,rows,columns,duplicate_keys,missing_values,sales_join_coverage_pct
0,market_signals,13200,14,0,0,100.0
1,market_activity,264000,16,0,0,100.0


In [52]:
print("Market signals sample")
display(
    market_signals.sample(5, random_state=RANDOM_SEED)
    .sort_values(["year_month", "product_group", "region_id"]))

print("Market activity sample")
display(
    market_activity.sample(5, random_state=RANDOM_SEED)
    .sort_values(["year_month", "product_id", "region_id"]))

campaign_summary = (
    market_activity.groupby("campaign_flag")
    .agg(
        rows=("product_id", "size"),
        average_spend_eur=("campaign_spend_eur", "mean"),
        average_website_visits=("website_visits", "mean"),
        average_demo_requests=("demo_requests", "mean"),
        average_mqls=("marketing_qualified_leads", "mean")).round(2))
campaign_summary

Market signals sample


,year_month,product_family,product_group,region_id,market_demand_index,market_growth_pct,competitor_pressure_index,seasonality_index,macro_business_index,supply_pressure_index,pipeline_interest_index,demand_shock_flag,market_opportunity_score,regional_market_growth_factor
4111,2021-08,IT Devices,laptop pro,2,91.43,-5.76,49.84,82.0,103.90,42.5,161.94,0,79.23,1.02
11786,2023-03,Services,service contract,10,125.96,9.42,59.17,100.0,105.38,42.5,0.00,0,67.99,0.95
10607,2023-12,Services,service contract,1,180.00,0.00,59.00,130.0,108.38,42.5,209.06,0,100.00,1.06
12227,2026-12,IT Devices,tablet enterprise,3,180.00,0.00,60.17,130.0,111.01,42.5,0.00,0,89.39,1.09
7372,2029-05,Legacy Products,legacy workstation,6,84.57,-0.38,60.68,100.0,112.03,42.5,0.00,0,51.10,1.14


Market activity sample


,year_month,product_id,region_id,campaign_flag,campaign_channel,campaign_spend_eur,campaign_impressions,campaign_clicks,campaign_ctr_pct,website_visits,product_page_views,demo_requests,marketing_qualified_leads,cost_per_lead_eur,regional_crm_activity_index,pipeline_interest_index
133585,2020-02,1101,3,0,No Active Campaign,0.00,0,0,0.000,252,436,6,6,0.00,100.00,0.00
171640,2023-05,1130,1,0,No Active Campaign,0.00,0,0,0.000,310,755,1,1,0.00,153.46,280.00
23678,2024-03,1017,10,1,Email,3129.40,92033,2133,2.318,2525,4951,40,179,17.48,37.40,0.00
20383,2024-08,1015,5,0,No Active Campaign,0.00,0,0,0.000,258,492,1,1,0.00,97.89,40.65
137085,2025-10,1103,9,1,Partner,1685.38,37635,1069,2.840,1493,3911,64,123,13.70,58.82,0.00


,rows,average_spend_eur,average_website_visits,average_demo_requests,average_mqls
campaign_flag,,,,,
0,220286,0.00,258.98,3.68,4.20
1,43714,4022.24,2308.86,48.19,165.03


## Export CSV files

In [ ]:
datasets = {
    "market_signals": market_signals,
    "market_activity": market_activity}

output_paths = {}
for dataset_name, dataset in datasets.items():
    output_path = OUTPUT_DIR / f"{dataset_name}.csv"
    dataset.to_csv(output_path, sep=",", decimal=".", index=False)
    output_paths[dataset_name] = output_path

assert set(output_paths) == {"factMarketSignals", "factMarketActivities"}

print("Created files:")
for dataset_name, output_path in output_paths.items():
    print(
        f"- {output_path.resolve()} "
        f"({len(datasets[dataset_name]):,} rows × {datasets[dataset_name].shape[1]} columns)")

Created files:
- C:\Users\Anast\OneDrive\Desktop\AS Portfolio\early-warning\data\new_generated_data\market_signals.csv (13,200 rows × 14 columns)
- C:\Users\Anast\OneDrive\Desktop\AS Portfolio\early-warning\data\new_generated_data\market_activity.csv (264,000 rows × 16 columns)
